# 📊 Resumo de Dados da Camada Bronze

## Objetivo
Este notebook automatiza a análise dos arquivos Parquet da camada Bronze, consolidando informações sobre:
- Tipos de dados e schema
- Contagem de registros e valores nulos
- Estatísticas descritivas (min, max, média, desvio padrão)
- Valores únicos e amostras
- Uso de memória

## Saída
Gera um arquivo `resumo_detalhado_bronze.parquet` com metadados consolidados para futura análise de qualidade dos dados.

In [1]:
import os
import pandas as pd
import geopandas as gpd
from pathlib import Path
from tqdm import tqdm
import numpy as np
from datetime import datetime

# Configuração de caminhos
BASE_DIR = Path("/home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios")
OUTPUT_DIR = BASE_DIR / "comex/data/resume"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

datasets_paths = {
    "comex_stat": BASE_DIR / "comex/data/bronze/comex_stat",
    "ibama_embargos": BASE_DIR / "ibama/data/bronze/ibama_embargos",
    "pam": BASE_DIR / "pam/data/bronze/pam",
    "pib_vab_agro": BASE_DIR / "pib/data/bronze/pib_vab_agro",
    "ppm_asininos": BASE_DIR / "ppm/data/bronze/ppm_asininos",
    "ppm_bovinos": BASE_DIR / "ppm/data/bronze/ppm_bovinos",
    "ppm_bubalinos": BASE_DIR / "ppm/data/bronze/ppm_bubalinos",
    "ppm_caprinos": BASE_DIR / "ppm/data/bronze/ppm_caprinos",
    "ppm_codornas": BASE_DIR / "ppm/data/bronze/ppm_codornas",
    "ppm_equinos": BASE_DIR / "ppm/data/bronze/ppm_equinos",
    "ppm_galinaceos_total": BASE_DIR / "ppm/data/bronze/ppm_galinaceos_total",
    "ppm_galinhas": BASE_DIR / "ppm/data/bronze/ppm_galinhas",
    "ppm_muar": BASE_DIR / "ppm/data/bronze/ppm_muar",
    "ppm_ovinos": BASE_DIR / "ppm/data/bronze/ppm_ovinos",
    "ppm_suinos_matrizes": BASE_DIR / "ppm/data/bronze/ppm_suinos_matrizes",
    "ppm_suinos_total": BASE_DIR / "ppm/data/bronze/ppm_suinos_total"
}

def convert_to_numpy_dtypes(df):
    """Converte colunas com dtype nullable do pandas para numpy dtype."""
    for col in df.columns:
        dtype = df[col].dtype
        # Converte StringDtype para object
        if isinstance(dtype, pd.StringDtype):
            df[col] = df[col].astype('object')
        # Converte dtypes nullable Int64, Float64 etc para numpy
        elif isinstance(dtype, pd.Int64Dtype):
            df[col] = df[col].astype('float64')
        elif isinstance(dtype, pd.Float64Dtype):
            df[col] = df[col].astype('float64')
    return df

def get_numeric_stats(series):
    """Calcula estatísticas descritivas para colunas numéricas."""
    stats = {
        "min": None,
        "max": None,
        "mean": None,
        "std": None,
        "median": None
    }
    
    # Verifica se é numérico
    if not pd.api.types.is_numeric_dtype(series):
        return stats
    
    try:
        min_val = series.min()
        max_val = series.max()
        
        if pd.notna(min_val):
            stats["min"] = float(min_val)
        if pd.notna(max_val):
            stats["max"] = float(max_val)
            
        mean_val = series.mean()
        if pd.notna(mean_val):
            stats["mean"] = float(mean_val)
            
        std_val = series.std()
        if pd.notna(std_val):
            stats["std"] = float(std_val)
            
        median_val = series.median()
        if pd.notna(median_val):
            stats["median"] = float(median_val)
    except Exception:
        pass
    
    return stats

def analyze_dataset(name, path):
    """Analisa um dataset e retorna resumo detalhado por coluna."""
    print(f"Analisando dataset: {name}")
    
    try:
        # Verifica se o caminho existe
        if not path.exists():
            print(f"  ⚠️ Caminho não encontrado: {path}")
            return None

        # Busca arquivos parquet/geoparquet
        files = list(path.glob("**/*.parquet")) + list(path.glob("**/*.geoparquet"))
        if not files:
            print(f"  ⚠️ Nenhum arquivo parquet encontrado para {name}")
            return None
        
        # Tenta carregar o dataset
        df = None
        load_error = None
        
        # Primeiro tenta com pandas
        try:
            df = pd.read_parquet(files[0])
        except Exception as e:
            load_error = str(e)
            # Se falhar, tenta geopandas
            try:
                df = gpd.read_parquet(files[0])
            except Exception as e2:
                print(f"  ❌ Erro ao ler arquivos para {name}: {load_error}")
                return None
        
        # Converte dtypes nullable para numpy
        df = convert_to_numpy_dtypes(df)
            
        # Metadados gerais
        num_rows = len(df)
        mem_usage = df.memory_usage(deep=True).sum() / (1024**2)
        
        if num_rows == 0:
            print(f"  ⚠️ Dataset {name} está vazio.")
            return None
        
        # Coleta metadados das colunas
        summary = []
        
        for col in df.columns:
            series = df[col]
            null_count = int(series.isna().sum())
            dtype_str = str(series.dtype)
            
            # Estatísticas base
            stats = {
                "dataset": name,
                "total_rows": num_rows,
                "mem_mb": round(mem_usage, 2),
                "column": col,
                "dtype": dtype_str,
                "non_null_count": int(series.count()),
                "null_count": null_count,
                "null_percent": round((null_count / num_rows) * 100, 2) if num_rows > 0 else 0,
                "unique_values": int(series.nunique()),
                "min": None,
                "max": None,
                "mean": None,
                "std": None,
                "median": None,
                "sample_values": "[]"
            }
            
            # Extrai amostra de valores (seguro para tipos não hashable)
            try:
                sample = series.dropna().unique()[:5].tolist()
                stats["sample_values"] = str(sample)
            except Exception:
                stats["sample_values"] = "Erro ao extrair amostra"
            
            # Calcula estatísticas numéricas (pula geometria)
            if "geometry" not in dtype_str.lower() and "geo" not in dtype_str.lower():
                numeric_stats = get_numeric_stats(series)
                stats.update(numeric_stats)
            
            summary.append(stats)
        
        print(f"  ✅ {num_rows} linhas, {len(df.columns)} colunas, {round(mem_usage, 2)} MB")
        return summary
    
    except Exception as e:
        print(f"  ❌ Erro ao analisar {name}: {str(e)}")
        return None

# Executa análise em todos os datasets
print(f"📅 Data da análise: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📁 Diretório de saída: {OUTPUT_DIR}\n")

all_summaries = []
for name, path in tqdm(datasets_paths.items(), desc="Progresso"):
    ds_summary = analyze_dataset(name, path)
    if ds_summary:
        all_summaries.extend(ds_summary)

# Exporta resultados
if all_summaries:
    summary_df = pd.DataFrame(all_summaries)
    
    # Salva em parquet
    output_file = OUTPUT_DIR / "resumo_detalhado_bronze.parquet"
    summary_df.to_parquet(output_file, index=False)
    
    print(f"\n{'='*60}")
    print(f"✅ Resumo exportado com sucesso!")
    print(f"   Arquivo: {output_file}")
    print(f"   Total de colunas analisadas: {len(summary_df)}")
    print(f"{'='*60}")
    
    # Exibe preview
    display(summary_df.head(10))
else:
    print("\n❌ Nenhum dado foi processado. Verifique se os caminhos estão corretos.")

📅 Data da análise: 2026-03-28 20:58:51
📁 Diretório de saída: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/comex/data/resume



Progresso:   0%|                                                                                                     | 0/16 [00:00<?, ?it/s]

Analisando dataset: comex_stat


Progresso:   6%|█████▊                                                                                       | 1/16 [00:00<00:14,  1.06it/s]

  ✅ 1603796 linhas, 11 colunas, 180.48 MB
Analisando dataset: ibama_embargos


...

Progresso:  69%|███████████████████████████████████████████████████████████████▎                            | 11/16 [00:02<00:00,  8.58it/s]

Progresso:  94%|██████████████████████████████████████████████████████████████████████████████████████▎     | 15/16 [00:02<00:00, 12.61it/s]

Progresso: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:02<00:00,  6.06it/s]

  ✅ 5568 linhas, 13 colunas, 4.02 MB
Analisando dataset: ppm_galinhas
  ✅ 5568 linhas, 13 colunas, 4.02 MB
Analisando dataset: ppm_muar
  ✅ 5568 linhas, 13 colunas, 4.02 MB
Analisando dataset: ppm_ovinos
  ✅ 5568 linhas, 13 colunas, 4.02 MB
Analisando dataset: ppm_suinos_matrizes
  ✅ 5568 linhas, 13 colunas, 4.02 MB
Analisando dataset: ppm_suinos_total
  ✅ 5568 linhas, 13 colunas, 4.02 MB

✅ Resumo exportado com sucesso!
   Arquivo: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/comex/data/resume/resumo_detalhado_bronze.parquet
   Total de colunas analisadas: 229


,dataset,total_rows,mem_mb,column,dtype,non_null_count,null_count,null_percent,unique_values,min,max,mean,std,median,sample_values
0,comex_stat,1603796,180.48,CO_ANO,int16,1603796,0,0.0,1,2024.0,2.024000e+03,2.024000e+03,0.000000e+00,2024.0,[2024]
1,comex_stat,1603796,180.48,CO_MES,int8,1603796,0,0.0,12,1.0,1.200000e+01,6.679265e+00,3.423077e+00,7.0,"[4, 11, 3, 8, 7]"
2,comex_stat,1603796,180.48,CO_NCM,int64,1603796,0,0.0,8019,1012100.0,9.706900e+07,4.997713e+07,3.185086e+07,48101399.0,"[94036000, 87088000, 2071400, 90303390, 40129090]"
3,comex_stat,1603796,180.48,CO_UNID,int64,1603796,0,0.0,13,10.0,2.200000e+01,1.067440e+01,1.753182e+00,10.0,"[11, 10, 13, 15, 16]"
4,comex_stat,1603796,180.48,CO_PAIS,int64,1603796,0,0.0,243,13.0,8.900000e+02,3.904963e+02,2.367281e+02,386.0,"[245, 493, 173, 158, 301]"
5,comex_stat,1603796,180.48,SG_UF_NCM,object,1603796,0,0.0,28,NaN,NaN,NaN,NaN,NaN,"['SP', 'MG', 'SC', 'CE', 'RS']"
6,comex_stat,1603796,180.48,CO_VIA,int64,1603796,0,0.0,12,0.0,1.500000e+01,3.001634e+00,2.625504e+00,1.0,"[4, 1, 7, 15, 0]"
7,comex_stat,1603796,180.48,CO_URF,int64,1603796,0,0.0,77,117600.0,9.999999e+06,7.988598e+05,3.603734e+05,817800.0,"[817600, 817800, 817700, 717800, 1017500]"
8,comex_stat,1603796,180.48,QT_ESTAT,int64,1603796,0,0.0,101177,0.0,1.452842e+10,3.497558e+05,3.782949e+07,32.0,"[8, 418615, 94080, 10, 26]"
9,comex_stat,1603796,180.48,KG_LIQUIDO,int64,1603796,0,0.0,123413,0.0,1.452842e+10,5.049486e+05,3.859872e+07,40.0,"[425, 60355, 94080, 12, 26]"
